## Extract Poses from Generated 3D WaymoSkelMo for FID 

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook
%matplotlib inline
%cd ..

import sys, os
import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from tqdm import tqdm


os.environ['PYOPENGL_PLATFORM'] = 'egl'

In [ ]:
# Choose the device to run the body model on.
comp_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
def copy2cpu(tensor):
    if isinstance(tensor, np.ndarray): return tensor
    return tensor.detach().cpu().numpy()

In [ ]:
from evaluation.tools.body_model import BodyModel

male_bm_path = 'evaluation/body_models/smplh/male/model.npz'
male_dmpl_path = 'evaluation/body_models/dmpls/male/model.npz'

female_bm_path = 'evaluation/body_models/smplh/female/model.npz'
female_dmpl_path = 'evaluation/body_models/dmpls/female/model.npz'

neutral_bm_path = 'evaluation/body_models/smplh/neutral/model.npz'
neutral_dmpl_path = 'evaluation/body_models/dmpls/neutral/model.npz'

num_betas = 10 # number of body parameters
num_dmpls = 8 # number of DMPL parameters

male_bm = BodyModel(bm_fname=male_bm_path, num_betas=num_betas, num_dmpls=num_dmpls, dmpl_fname=male_dmpl_path).to(comp_device)
faces = copy2cpu(male_bm.f)

female_bm = BodyModel(bm_fname=female_bm_path, num_betas=num_betas, num_dmpls=num_dmpls, dmpl_fname=female_dmpl_path).to(comp_device)

neutral_bm = BodyModel(bm_fname=neutral_bm_path, num_betas=num_betas, num_dmpls=num_dmpls, dmpl_fname=neutral_dmpl_path).to(comp_device)

In [ ]:
paths = []
folders = []
dataset_names = []
# for root, dirs, files in os.walk('./waymo_filtered_150_relatives'):
for root, dirs, files in os.walk('/dataset/guangxun/test/generated/final_30fps'):
    folders.append(root)
    for name in files:
        dataset_name = root.split('/')[-3]
        if dataset_name not in dataset_names:
            dataset_names.append(dataset_name)
        paths.append(os.path.join(root, name))

In [ ]:
# save_root = './pose_data_waymo_150_relatives'
save_root = 'FID_data/pose_data'
# save_folders = [folder.replace('./waymo_filtered_150_relatives', './pose_data_waymo_150_relatives') for folder in folders]

save_folders = [folder.replace('/dataset/guangxun/test/generated/final_30fps', save_root) for folder in folders]
for folder in save_folders:
    os.makedirs(folder, exist_ok=True)
group_path = [[path for path in paths if name in path] for name in dataset_names]

In [ ]:
trans_matrix = np.array([[1.0, 0.0, 0.0],
                            [0.0, 0.0, 1.0],
                            [0.0, 1.0, 0.0]])
ex_fps = 30
def amass_to_pose(src_path, save_path):
    bdata = np.load(src_path, allow_pickle=True)
    fps = 30

    bm = neutral_bm
    trans = bdata['trans']

    trans[:,:] -= trans[0,:]
    trans[:,2] += 0.85
    N_raw = len(trans)

    pose_body = bdata['pose_body'][:, :63]
    root_orient = bdata['root_orient']




    clip_frames = 196
    overlap = 2

    # clip segmentation
    step = clip_frames - overlap
    clip_idx = 0  
    
    for i in range(0, N_raw, step):
        end = i + clip_frames
        if end <= N_raw:
            clip_indices = np.arange(i, end)
            valid_indices = np.arange(len(clip_indices))
        else:
            clip_indices = np.arange(i, N_raw)

        clip_pose_body = pose_body[clip_indices]
        clip_trans = trans[clip_indices]
        clip_root_orient = root_orient[clip_indices]

        if clip_trans.size > 0:
            body_parms = {
                    'root_orient': torch.Tensor(clip_root_orient).to(comp_device),
                    'pose_body': torch.Tensor(clip_pose_body).to(comp_device),
                    'pose_hand': torch.Tensor(np.zeros((clip_pose_body.shape[0], 90))).to(comp_device),
                    'trans': torch.Tensor(clip_trans).to(comp_device),
                    # 'betas': torch.Tensor(bdata['betas']).to(comp_device),
                    # 'betas': torch.Tensor(np.repeat(bdata['betas'][0][np.newaxis], repeats=len(clip_trans), axis=0)).to(comp_device),
                    'betas': torch.Tensor(np.zeros((len(clip_trans), 10))).to(comp_device),

                }

            with torch.no_grad():
                body = bm(**body_parms)
            pose_seq_np = body.Jtr.detach().cpu().numpy()
            pose_seq_np_n = np.dot(pose_seq_np, trans_matrix)
            
            new_save_path = save_path[:-4]+'_' + str(clip_idx) + '.npy'

            np.save(new_save_path, pose_seq_np_n)

        else:
            print(f"Clip {clip_idx} has no valid frames, skipping...")

        clip_idx += 1  # 更新片段计数器

    return fps

In [ ]:
group_path = group_path
all_count = sum([len(paths) for paths in group_path])
cur_count = 0

In [ ]:
import time
for paths in group_path:
    dataset_name = paths[0].split('/')[2]
    pbar = tqdm(paths)
    pbar.set_description('Processing: %s'%dataset_name)
    fps = 0
    for path in pbar:
        # save_path = path.replace('./waymo_filtered_150_relatives', './pose_data_waymo_150_relatives')
        save_path = path.replace('/dataset/guangxun/test/generated/final_30fps', 'FID_data/pose_data')

        save_path = save_path[:-3] + 'npy'
        fps = amass_to_pose(path, save_path)
        
    cur_count += len(paths)
    print('Processed / All (fps %d): %d/%d'% (fps, cur_count, all_count) )
    time.sleep(0.5)

In [ ]:
import os
import shutil

# copy final pose data to waymo_joints 
source_dir = './FID_data/pose_data'
target_dir = './FID_data/waymo_joints'

os.makedirs(target_dir, exist_ok=True)

for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.endswith('.npy'):

            source_file = os.path.join(root, file)
            target_file = os.path.join(target_dir, file)
            
            shutil.copy2(source_file, target_file)
            print(f'Copied {source_file} to {target_file}')